# NN_06 — FiLM ablation

`BUILD_SPEC.md` §7, row 06, **the decisive notebook**: FiLM vs FiLM-removed, same seeds, same cycles, paired per cycle. Acceptance: a paired per-cycle difference and a p-value. This is the notebook that answers the question the experiment was built to ask (`OUTLINE.md` §1): does macro conditioning through FiLM express anything that shows up as out-of-sample IC?

**Protocol — identical to `NN_05` in every respect, fixed before running:** graded ListNet target at temperature 0.5; CPU; seeds `(0, 1, 2, 3, 4)`; the same 9 target cycles the baseline trained (8 with a measured IC); the same production rolling window with its most recent quarter held out for early stopping; the same full-batch AdamW (lr 1e-3, weight decay 1e-2), 1,000-step cap, validation every 10 steps, patience 20, best-validation weights restored; the same Materials-excluded scoring universe and forward returns; every IC through `spearman_correlation`. The two arms are the same `NeuralRanker` class with `use_film=True` and `use_film=False`. Because both arms construct identical modules and consume the random stream identically, the same seed gives **bit-identical initial weights and identical dropout masks** in both arms — verified below — so the only difference between paired fits is whether `h' = (1 + γ) ⊙ h + β` is applied.

**Pairing.** Differences are taken per (cycle, seed) — FiLM IC minus no-FiLM IC on the same cycle, the same names, the same returns, the same seed — then averaged over seeds within each cycle. The primary test is a paired *t*-test across the 8 measured cycles on those seed-mean differences (df = 7, two-sided). Per-seed paired tests and the spread of the effect across seeds are reported so the effect can be read against seed noise.

**Verdict rule, registered before the results were seen:** FiLM *helps* on this slice if the mean paired difference is > 0 with p < 0.05; *hurts* if it is < 0 with p < 0.05; otherwise the result is *null on this slice*. In every case the effect is also compared with the seed noise. **Eight measured cycles is thin** — far thinner than the 22-cycle full-history baseline — and the conditioner sees only 4–12 distinct macro states per training window (`NN_03`); a null here means "not distinguishable from zero at this sample size", not "macro conditioning is useless". **No production value is claimed for either arm.**

**Known caveat carried in from `NN_05`:** under this protocol early stopping selects models 10–50 steps from initialization in both arms; the ablation is run under the protocol as fixed, and the best-step distributions of both arms are reported.

**Data mode: REAL DATA.** Everything below is computed from the cached real-data slice (`rd.slice_bundle()`, fingerprint verified in `NN_00`) and from `NN_05`'s saved run artifact, whose universe, returns and provenance are verified against the slice before use. No synthetic, simulated, or approximated value appears anywhere; if the raw exports are absent, every computing cell prints the explicit unavailability notice and does nothing.

Model code lives **in this notebook**, identical to `NN_05`'s. Kernel: `atlasquant-venv` (= `.venv/bin/python`).

In [1]:
import nn_common as nc

rd = nc.rd
AVAILABLE = nc.provenance_banner("NN_06_film_ablation.ipynb")

neural_ranker / NN_06_film_ablation.ipynb
DATA MODE: REAL DATA.

Source: Bloomberg CSV exports under this repository's gitignored raw-data
root for multi_factor_ranking_ml -- 1,520 tickers, quarterly fundamentals
from 1980 onward, plus daily closes and macro series.

Every number below is computed from that real data through production
AtlasQuant code. No synthetic, simulated, imputed, or approximated values
are used anywhere in this notebook series.

Bounded slice: quarterly evaluation cycles 2020-10-01 .. 2024-01-01
(a genuine subset of the history, chosen so each notebook runs quickly --
not a downsample or an approximation of a longer run).

SOURCE MANIFEST (SHA-256 of every raw CSV the pipeline reads)
  raw root: /Users/gavinhussey/Downloads/AtlasQuant/data/raw/multi_factor_ranking_ml
  fundamentals_quarterly.csv      281.3 MB  2026-08-31 23:01:10  3315e7104e640fc790b6e12c66b0f6a122cf66bb97a8ed3124abaced0c86b48a
  Close_Price.csv                  67.6 MB  2026-08-31 23:07:42  2723

libraries         : python=3.14.5, torch=2.13.0, numpy=2.5.2, pandas=3.0.5, scipy=1.18.0, lightgbm=4.7.0


## The model and the objective — the `NN_05` definitions, verbatim

In [2]:
import copy
import json
import time
from datetime import datetime, time as _time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

N_FEATURES = len(nc.STOCK_CONTINUOUS)   # 63
N_MACRO = len(nc.MACRO)                 # 6
SECTOR_VOCAB, SECTOR_DIM = len(nc.SECTOR_VOCABULARY), 8
QUARTER_VOCAB, QUARTER_DIM = 4, 4
HIDDEN, LATENT = 128, 64
N_GRADES = rd.config().n_relevance_grades   # 10
TEMPERATURE = 0.5   # fixed protocol decision (NN_05); never tuned


class NeuralRanker(nn.Module):
    def __init__(self, use_film: bool = True, dropout: float = 0.3):
        super().__init__()
        self.use_film = use_film
        self.sector_emb = nn.Embedding(SECTOR_VOCAB, SECTOR_DIM)
        self.quarter_emb = nn.Embedding(QUARTER_VOCAB, QUARTER_DIM)
        self.encoder = nn.Sequential(
            nn.Linear(2 * N_FEATURES + SECTOR_DIM + QUARTER_DIM, HIDDEN),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(HIDDEN),
            nn.Linear(HIDDEN, LATENT),
            nn.GELU(),
        )
        self.film = nn.Sequential(nn.Linear(N_MACRO, 32), nn.GELU(), nn.Linear(32, 2 * LATENT))
        nn.init.normal_(self.film[-1].weight, std=1e-4)
        nn.init.zeros_(self.film[-1].bias)
        self.head = nn.Sequential(nn.Linear(LATENT, 32), nn.GELU(), nn.Linear(32, 1))

    def film_params(self, m):
        gb = self.film(m)
        return gb[:LATENT], gb[LATENT:]

    def forward(self, X, mask, sector, quarter, m):
        z = torch.cat([X, mask, self.sector_emb(sector), self.quarter_emb(quarter - 1)], dim=1)
        h = self.encoder(z)
        if self.use_film:
            gamma, beta = self.film_params(m)
            h = (1.0 + gamma) * h + beta
        return self.head(h).squeeze(-1)


def scaled_gain(y):
    return (2.0 ** y.to(torch.float32) - 1.0) / (2.0 ** (N_GRADES - 1) - 1.0)


def graded_target(y, temperature=TEMPERATURE):
    return torch.softmax(scaled_gain(y) / temperature, dim=0)


def listnet_loss(scores, y):
    return -(graded_target(y) * F.log_softmax(scores, dim=0)).sum()


def score_cycle(net, inputs):
    return net(inputs.X, inputs.mask, inputs.sector, inputs.quarter, inputs.m)

## Inputs: the slice, and `NN_05`'s verified run artifact

`NN_05` saved, per measured cycle, the baseline's Materials-excluded scored universe (symbols), the real clipped forward returns and the baseline scores, plus its own per-fit records. Before any of it is used: the artifact's slice fingerprint and config identity must equal this checkout's; its recorded protocol (temperature, device, seeds) must equal this notebook's; the symbols must map one-to-one onto the cycle's instruments; and the baseline IC recomputed from the artifact's returns and baseline scores must equal `bundle.ic_result`'s stored IC bit for bit on all 8 cycles — the slice's own numbers, independent of the artifact, are the cross-check.

In [3]:
if rd.require_data():
    from atlas_quant.backtest.multi_factor_ranking_runner import decile_spread, spearman_correlation
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import TrainingState

    IC_CALLS = 0

    def ic_of(scores, returns):
        global IC_CALLS
        IC_CALLS += 1
        return spearman_correlation(list(scores), list(returns))

    bundle = rd.slice_bundle()
    cfg = rd.config()
    artifact_path = nc.NEURAL_RANKER_DIR / "outputs" / "NN_05_walkforward.json"
    NN05 = json.loads(artifact_path.read_text())
    print("artifact          ", artifact_path, f"({artifact_path.stat().st_size / 1e6:.2f} MB)")
    print("artifact notebook ", NN05["notebook"], "| git", NN05["git_commit"])
    print("artifact protocol ", NN05["protocol"])
    assert NN05["slice_fingerprint"] == bundle.fingerprint, "artifact was built from a different slice"
    assert NN05["config_identity"] == cfg.identity()
    assert NN05["protocol"]["temperature"] == TEMPERATURE and NN05["device"] == "cpu" and tuple(NN05["seeds"]) == nc.SEEDS
    assert (NN05["protocol"]["lr"], NN05["protocol"]["weight_decay"], NN05["protocol"]["max_steps"],
            NN05["protocol"]["eval_every"], NN05["protocol"]["patience"]) == (1e-3, 1e-2, 1000, 10, 20)

    DEVICE = torch.device("cpu")
    ALL = nc.bundle_cycle_tensors(bundle, transform=nc.normalize_cycle)
    MI = {q: nc.to_model_input(t, DEVICE) for q, t in ALL.items()}
    for q, inputs in MI.items():
        for name in ("X", "mask", "sector", "quarter", "m"):
            assert bool(torch.isfinite(getattr(inputs, name)).all())
        assert np.array_equal(inputs.mask.numpy().astype(bool), np.isnan(ALL[q].X))

    baseline = {r.cycle.quarter_start: r for r in bundle.ic_result.cycle_results}
    TRAINED = [r.cycle for r in bundle.ic_result.cycle_results if r.training_state == TrainingState.TRAINED]
    MEASURED = [r.cycle for r in bundle.ic_result.cycle_results if r.ic is not None]
    UNIVERSE = {}
    print(f"\n{'cycle':<12}{'scored':>7}{'stored':>7}{'stored IC':>11}{'from artifact':>15}{'equal':>7}{'Materials':>10}")
    for cycle in MEASURED:
        q = cycle.quarter_start
        entry = NN05["universe"][q.isoformat()]
        by_symbol = {iid.symbol: iid for iid in ALL[q].instrument_ids}
        assert len(by_symbol) == ALL[q].n, "symbols must be unique within the cycle"
        ids = [by_symbol[s] for s in entry["symbols"]]
        rets, base_scores = entry["returns"], entry["baseline_scores"]
        recomputed = ic_of(base_scores, rets)
        obs_by_id = {o.instrument_id: o for o in bundle.feature_results[q].observations}
        materials = sum(1 for i in ids if obs_by_id[i].sector == "Materials")
        print(f"{q.isoformat():<12}{len(ids):>7}{baseline[q].scored_for_ic_count:>7}{baseline[q].ic:>+11.4f}{recomputed:>+15.4f}{str(recomputed == baseline[q].ic):>7}{materials:>10}")
        assert recomputed == baseline[q].ic and len(ids) == baseline[q].scored_for_ic_count and materials == 0
        assert abs(NN05["baseline_ic"][q.isoformat()] - baseline[q].ic) == 0
        UNIVERSE[q] = (ids, rets, base_scores)
    NN05_RECORDS = {(r["target"], r["seed"]): r for r in NN05["results"]}
    print(f"\nartifact verified against the slice on all {len(UNIVERSE)} measured cycles; {len(NN05_RECORDS)} NN_05 fit records loaded")
    print("device used:", DEVICE, "| torch threads:", torch.get_num_threads())

artifact           /Users/gavinhussey/Downloads/AtlasQuant/neural_ranker/outputs/NN_05_walkforward.json (1.73 MB)
artifact notebook  NN_05_walkforward.ipynb | git 7b7f2681146b8ed0a65ca808697a6194296a781f
artifact protocol  {'lr': 0.001, 'weight_decay': 0.01, 'max_steps': 1000, 'eval_every': 10, 'patience': 20, 'temperature': 0.5, 'target': 'softmax(((2**y - 1) / (2**9 - 1)) / 0.5)', 'holdout': 'most recent included training quarter'}



cycle        scored stored  stored IC  from artifact  equal Materials
2022-01-01     1370   1370    -0.0261        -0.0261   True         0
2022-04-01     1372   1372    -0.1900        -0.1900   True         0
2022-07-01     1374   1374    +0.0618        +0.0618   True         0
2022-10-01     1376   1376    +0.0583        +0.0583   True         0
2023-01-01     1381   1381    +0.0708        +0.0708   True         0
2023-04-01     1382   1382    +0.0335        +0.0335   True         0
2023-07-01     1391   1391    +0.0877        +0.0877   True         0
2023-10-01     1401   1401    +0.1240        +0.1240   True         0

artifact verified against the slice on all 8 measured cycles; 45 NN_05 fit records loaded
device used: cpu | torch threads: 8


## Training window and the paired fit

`training_window` is `NN_05`'s, with the same point-in-time assertions inside it. `fit_and_score` takes the arm as an argument and is otherwise `NN_05`'s fit, so a FiLM fit here must reproduce `NN_05`'s saved record bit for bit — asserted for every (cycle, seed) as the run proceeds.

In [4]:
if rd.require_data():
    from atlas_quant.strategies.multi_factor_ranking_ml.config import FEATURE_SCHEMA_VERSION
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import (
        build_training_dataset,
        check_training_eligibility,
    )

    LR, WEIGHT_DECAY = 1e-3, 1e-2
    MAX_STEPS, EVAL_EVERY, PATIENCE = 1000, 10, 20

    def training_window(target):
        cutoff = datetime.combine(target.cutoff, _time.min)
        dataset = build_training_dataset(
            target.quarter_start, cutoff, bundle.labeled_quarters,
            strategy_id=cfg.strategy_id, feature_schema_version=FEATURE_SCHEMA_VERSION,
            ml_train_years=cfg.ml_train_years, model_config_identity=cfg.model.identity(),
        )
        assert check_training_eligibility(dataset, min_train_quarters=cfg.min_train_quarters).eligible
        included = list(dataset.included_quarters)
        assert dataset.groups == tuple(ALL[q].n for q in included)
        assert dataset.instrument_ids == tuple(i for q in included for i in ALL[q].instrument_ids)
        val_q, train_qs = included[-1], included[:-1]
        window_start = target.quarter_start.replace(year=target.quarter_start.year - cfg.ml_train_years)
        for q in train_qs + [val_q]:
            assert q < target.quarter_start and ALL[q].label_available_at < cutoff and q >= window_start
        assert val_q == max(included) and target.quarter_start not in included
        return train_qs, val_q, cutoff

    def fit_and_score(target, seed, use_film):
        train_qs, val_q, cutoff = training_window(target)
        nc.seed_everything(seed)
        net = NeuralRanker(use_film=use_film).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        best_val, best_state, best_step, bad, steps = float("inf"), None, 0, 0, 0
        t0 = time.perf_counter()
        for step in range(1, MAX_STEPS + 1):
            net.train()
            opt.zero_grad()
            loss = torch.stack([listnet_loss(score_cycle(net, MI[q]), MI[q].y) for q in train_qs]).mean()
            loss.backward()
            opt.step()
            steps = step
            if step % EVAL_EVERY == 0:
                net.eval()
                with torch.no_grad():
                    val = listnet_loss(score_cycle(net, MI[val_q]), MI[val_q].y).item()
                if val < best_val:
                    best_val, best_state, best_step, bad = val, copy.deepcopy(net.state_dict()), step, 0
                else:
                    bad += 1
                    if bad >= PATIENCE:
                        break
        net.load_state_dict(best_state)
        net.eval()
        with torch.no_grad():
            scores_all = score_cycle(net, MI[target.quarter_start]).numpy()
            gamma, beta = net.film_params(MI[target.quarter_start].m)
        assert scores_all.shape == (ALL[target.quarter_start].n,) and np.isfinite(scores_all).all()
        record = {
            "arm": "film" if use_film else "no_film", "seed": seed, "target": target.quarter_start.isoformat(),
            "train_quarters": [q.isoformat() for q in train_qs], "validation_quarter": val_q.isoformat(),
            "steps": steps, "best_step": best_step, "best_val_loss": best_val, "final_train_loss": loss.item(),
            "gamma_max": float(gamma.abs().max()), "beta_max": float(beta.abs().max()),
            "scored_count": 0, "ic": None, "decile_spread": None, "seconds": time.perf_counter() - t0,
        }
        if target.quarter_start in UNIVERSE:
            ids, rets, _ = UNIVERSE[target.quarter_start]
            position = {iid: k for k, iid in enumerate(ALL[target.quarter_start].instrument_ids)}
            nn_scores = [float(scores_all[position[i]]) for i in ids]
            assert len(nn_scores) == len(ids) == baseline[target.quarter_start].scored_for_ic_count
            record["scored_count"] = len(nn_scores)
            record["ic"] = ic_of(nn_scores, rets)
            record["decile_spread"] = decile_spread(list(zip(nn_scores, rets)))
            record["scores"] = nn_scores
        return record

### The two arms start identical

Same seed, both arms: every shared parameter tensor must be bit-identical at initialization, and the unused FiLM module is constructed identically too, so the random stream is consumed identically. At initialization the FiLM arm's γ and β are ~10⁻³, so the two arms' first forward passes differ only by that near-zero modulation.

In [5]:
if rd.require_data():
    nc.seed_everything(nc.SEEDS[0]); a = NeuralRanker(use_film=True).to(DEVICE)
    nc.seed_everything(nc.SEEDS[0]); b = NeuralRanker(use_film=False).to(DEVICE)
    sa, sb = a.state_dict(), b.state_dict()
    identical = all(torch.equal(sa[k], sb[k]) for k in sa) and sa.keys() == sb.keys()
    print("state_dict keys:", len(sa), "| all parameter tensors bit-identical across arms at init:", identical)
    q = TRAINED[0].quarter_start
    a.eval(); b.eval()
    with torch.no_grad():
        gamma, beta = a.film_params(MI[q].m)
        diff = (score_cycle(a, MI[q]) - score_cycle(b, MI[q])).abs().max().item()
    print(f"init on {q}: max|gamma| {gamma.abs().max().item():.2e}, max|beta| {beta.abs().max().item():.2e}, max |score_film - score_nofilm| {diff:.2e}")
    assert identical and gamma.abs().max().item() < 1e-2 and beta.abs().max().item() < 1e-2

state_dict keys: 16 | all parameter tensors bit-identical across arms at init: True
init on 2022-01-01: max|gamma| 3.53e-03, max|beta| 3.57e-03, max |score_film - score_nofilm| 2.74e-04


## The paired run

For each target cycle and seed, the FiLM arm is fitted and checked against `NN_05`'s saved record (IC, best validation loss, best step, steps, scores — all bit for bit), then the no-FiLM arm is fitted with the same seed.

In [6]:
if rd.require_data():
    RESULTS = []
    print(f"device {DEVICE} | temperature {TEMPERATURE} | AdamW lr {LR} wd {WEIGHT_DECAY} | max {MAX_STEPS} steps, validate every {EVAL_EVERY}, patience {PATIENCE} | seeds {nc.SEEDS}")
    print(f"{'target':<12}{'seed':>5}{'train q':>8}{'FiLM best@':>11}{'FiLM val':>10}{'FiLM IC':>9}{'noFiLM best@':>13}{'noFiLM val':>11}{'noFiLM IC':>10}{'diff':>9}{'sec':>7}")
    mismatches = 0
    for target in TRAINED:
        for seed in nc.SEEDS:
            f = fit_and_score(target, seed, use_film=True)
            saved = NN05_RECORDS[(f["target"], seed)]
            same = (
                f["ic"] == saved["ic"] and f["best_val_loss"] == saved["best_val_loss"] and f["best_step"] == saved["best_step"]
                and f["steps"] == saved["steps"] and f.get("scores") == saved.get("scores")
            )
            mismatches += (not same)
            n = fit_and_score(target, seed, use_film=False)
            RESULTS.extend([f, n])
            d = (f["ic"] - n["ic"]) if f["ic"] is not None else None
            fmt = lambda v: f"{v:+.4f}" if v is not None else "   -   "
            print(f"{f['target']:<12}{seed:>5}{len(f['train_quarters']):>8}{f['best_step']:>11}{f['best_val_loss']:>10.5f}{fmt(f['ic']):>9}"
                  f"{n['best_step']:>13}{n['best_val_loss']:>11.5f}{fmt(n['ic']):>10}{fmt(d):>9}{f['seconds'] + n['seconds']:>7.1f}")
    total = sum(r["seconds"] for r in RESULTS)
    print(f"\nfits: {len(RESULTS)} ({len(TRAINED)} cycles x {len(nc.SEEDS)} seeds x 2 arms); total {total:.0f}s; mean {total / len(RESULTS):.1f}s per fit")
    print(f"FiLM-arm records that differ from NN_05's saved records: {mismatches}")
    assert mismatches == 0, "the FiLM arm must reproduce NN_05 bit for bit"

device cpu | temperature 0.5 | AdamW lr 0.001 wd 0.01 | max 1000 steps, validate every 10, patience 20 | seeds (0, 1, 2, 3, 4)
target       seed train q FiLM best@  FiLM val  FiLM IC noFiLM best@ noFiLM val noFiLM IC     diff    sec


2022-01-01      0       3         20   7.28509  +0.0732           10    7.28169   +0.0844  -0.0112    3.9


2022-01-01      1       3         20   7.28336  +0.0613           10    7.29174   +0.1845  -0.1231    4.0


2022-01-01      2       3         20   7.29062  +0.0783           10    7.29689   +0.1142  -0.0359    4.0


2022-01-01      3       3         20   7.28209  +0.0998           10    7.28294   +0.1333  -0.0335    4.0


...

2024-01-01      0      11         30   7.28033     -              40    7.28167      -        -      16.9


2024-01-01      1      11         30   7.27463     -              30    7.27106      -        -      16.6


2024-01-01      2      11         20   7.27750     -              40    7.26872      -        -      16.6


2024-01-01      3      11         40   7.27432     -              40    7.27318      -        -      17.3


2024-01-01      4      11         30   7.27171     -              30    7.27270      -        -      16.7

fits: 90 (9 cycles x 5 seeds x 2 arms); total 451s; mean 5.0s per fit
FiLM-arm records that differ from NN_05's saved records: 0


## Paired results

Per cycle: the seed-mean IC of each arm, the seed-mean paired difference, its range across the five seeds, and how many of the five seeds favour FiLM. Then the per-seed view, the primary paired *t*-test across the 8 cycles, the seed-noise scale, the early-stopping distributions of both arms, and the verdict under the rule registered above.

In [7]:
if rd.require_data():
    from scipy import stats

    measured_keys = [c.quarter_start.isoformat() for c in MEASURED]
    nn_measured = sorted({r["target"] for r in RESULTS if r["ic"] is not None})
    assert nn_measured == measured_keys, "measured cycles must match the baseline's 8 exactly"
    rec = {(r["arm"], r["target"], r["seed"]): r for r in RESULTS}

    film_sm, nofilm_sm, diff_sm, diff_by_seed = {}, {}, {}, {s: [] for s in nc.SEEDS}
    print(f"{'cycle':<12}{'LightGBM':>10}{'FiLM':>9}{'no-FiLM':>9}{'FiLM - noFiLM':>15}{'diff min':>10}{'diff max':>10}{'seeds FiLM>':>12}")
    for k in measured_keys:
        f_ics = [rec[("film", k, s)]["ic"] for s in nc.SEEDS]
        n_ics = [rec[("no_film", k, s)]["ic"] for s in nc.SEEDS]
        d = [a - b for a, b in zip(f_ics, n_ics)]
        for s, v in zip(nc.SEEDS, d):
            diff_by_seed[s].append(v)
        film_sm[k], nofilm_sm[k], diff_sm[k] = float(np.mean(f_ics)), float(np.mean(n_ics)), float(np.mean(d))
        print(f"{k:<12}{baseline[datetime.fromisoformat(k).date()].ic:>+10.4f}{film_sm[k]:>+9.4f}{nofilm_sm[k]:>+9.4f}{diff_sm[k]:>+15.4f}{min(d):>+10.4f}{max(d):>+10.4f}{sum(v > 0 for v in d):>9} of 5")

    D = np.array([diff_sm[k] for k in measured_keys])
    t_stat, p_value = stats.ttest_rel([film_sm[k] for k in measured_keys], [nofilm_sm[k] for k in measured_keys])
    print(f"\nprimary paired t-test, seed-mean FiLM IC vs seed-mean no-FiLM IC across {len(D)} cycles (df = {len(D) - 1}, two-sided):")
    print(f"  mean difference {D.mean():+.4f}   std (ddof=1) {D.std(ddof=1):.4f}   SE {D.std(ddof=1) / np.sqrt(len(D)):.4f}   t = {t_stat:+.3f}   p = {p_value:.3f}   FiLM higher in {int((D > 0).sum())} of {len(D)} cycles")

    print(f"\n{'seed':<6}{'mean FiLM IC':>13}{'mean no-FiLM IC':>16}{'mean diff':>11}{'t':>8}{'p':>8}{'cycles FiLM>':>13}")
    per_seed_mean_diff = []
    for s in nc.SEEDS:
        fi = [rec[("film", k, s)]["ic"] for k in measured_keys]
        ni = [rec[("no_film", k, s)]["ic"] for k in measured_keys]
        ts, ps = stats.ttest_rel(fi, ni)
        per_seed_mean_diff.append(float(np.mean(diff_by_seed[s])))
        print(f"{s:<6}{np.mean(fi):>+13.4f}{np.mean(ni):>+16.4f}{np.mean(diff_by_seed[s]):>+11.4f}{ts:>+8.3f}{ps:>8.3f}{sum(v > 0 for v in diff_by_seed[s]):>9} of {len(measured_keys)}")
    seed_noise = float(np.std(per_seed_mean_diff, ddof=1))
    within_cycle_seed_std = float(np.mean([np.std([rec[('film', k, s)]['ic'] - rec[('no_film', k, s)]['ic'] for s in nc.SEEDS], ddof=1) for k in measured_keys]))
    print(f"\nseed variance: std across seeds of the per-seed mean difference {seed_noise:.4f}; "
          f"mean within-cycle std of the difference across seeds {within_cycle_seed_std:.4f}; "
          f"per-seed mean difference min {min(per_seed_mean_diff):+.4f}, max {max(per_seed_mean_diff):+.4f}")
    print(f"effect / seed-noise ratio: {abs(D.mean()) / seed_noise:.2f}" if seed_noise > 0 else "seed noise is zero")

    for arm in ("film", "no_film"):
        bs = [r["best_step"] for r in RESULTS if r["arm"] == arm]
        st = [r["steps"] for r in RESULTS if r["arm"] == arm]
        print(f"early stopping, {arm:<8}: best step min {min(bs)}, median {int(np.median(bs))}, max {max(bs)}; steps run {min(st)}..{max(st)}; at cap {sum(1 for v in st if v == MAX_STEPS)}")
    g = [r["gamma_max"] for r in RESULTS if r["arm"] == "film"]
    print(f"FiLM arm, |gamma| max at the scored target across the {len(g)} fits: min {min(g):.4f}, median {np.median(g):.4f}, max {max(g):.4f}")

    if D.mean() > 0 and p_value < 0.05:
        VERDICT = "FiLM HELPS on this slice"
    elif D.mean() < 0 and p_value < 0.05:
        VERDICT = "FiLM HURTS on this slice"
    else:
        VERDICT = "NULL on this slice: the FiLM effect is not distinguishable from zero at 8 cycles"
    print(f"\nVERDICT (rule registered before the run): {VERDICT}")
    expected_ic_calls = len(UNIVERSE) + sum(1 for r in RESULTS if r["ic"] is not None)
    print(f"IC computations, all via spearman_correlation: {IC_CALLS} (expected {expected_ic_calls})")
    assert IC_CALLS == expected_ic_calls

cycle         LightGBM     FiLM  no-FiLM  FiLM - noFiLM  diff min  diff max seeds FiLM>
2022-01-01     -0.0261  +0.0759  +0.1179        -0.0420   -0.1231   -0.0062        0 of 5
2022-04-01     -0.1900  -0.1422  -0.1002        -0.0420   -0.1115   -0.0027        0 of 5
2022-07-01     +0.0618  +0.1046  +0.1044        +0.0002   -0.0085   +0.0070        3 of 5
2022-10-01     +0.0583  +0.0633  +0.0641        -0.0008   -0.0089   +0.0036        3 of 5
2023-01-01     +0.0708  +0.0584  +0.0473        +0.0111   -0.0454   +0.0999        2 of 5
2023-04-01     +0.0335  +0.0874  +0.0974        -0.0099   -0.0700   +0.0232        2 of 5
2023-07-01     +0.0877  +0.1029  +0.0985        +0.0044   -0.0022   +0.0128        4 of 5
2023-10-01     +0.1240  +0.0436  +0.0399        +0.0037   -0.0231   +0.0203        3 of 5

primary paired t-test, seed-mean FiLM IC vs seed-mean no-FiLM IC across 8 cycles (df = 7, two-sided):
  mean difference -0.0094   std (ddof=1) 0.0210   SE 0.0074   t = -1.271   p = 0.244   Fi

## Point-in-time check

Stated explicitly, per `CLAUDE.md` and `BUILD_SPEC.md` §6, and asserted inside every fit rather than assumed: every training and validation quarter is strictly before its target and inside the trailing window; every label used was available strictly before the target's cutoff; the validation quarter is the most recent quarter of the production window; the target quarter never appears in the window; normalization and the input transform are the per-cycle policies audited in `NN_02`–`NN_04`; FiLM sees only the target cycle's own `m` at scoring time; forward returns enter only the IC, never the model. The artifact from `NN_05` contributed only the scoring universe and returns, which were verified against the slice's own stored ICs before use.

In [8]:
if rd.require_data():
    checked = 0
    for r in RESULTS:
        target = datetime.fromisoformat(r["target"]).date()
        cutoff = datetime.combine(next(c for c in TRAINED if c.quarter_start == target).cutoff, _time.min)
        used = [datetime.fromisoformat(q).date() for q in r["train_quarters"]] + [datetime.fromisoformat(r["validation_quarter"]).date()]
        assert target not in used and all(q < target for q in used)
        assert all(ALL[q].label_available_at < cutoff for q in used)
        assert datetime.fromisoformat(r["validation_quarter"]).date() == max(used)
        checked += 1
    print(f"records checked: {checked} (both arms); training/validation quarters before target, labels available before cutoff, validation = most recent window quarter: True")
    print("point-in-time checks: passed")

records checked: 90 (both arms); training/validation quarters before target, labels available before cutoff, validation = most recent window quarter: True
point-in-time checks: passed


## Saved output

Both arms' per-fit records (with scores for the measured cycles) for `NN_07`/`NN_08`. Gitignored, reproducible bit for bit on CPU.

In [9]:
if rd.require_data():
    out_path = nc.NEURAL_RANKER_DIR / "outputs" / "NN_06_film_ablation.json"
    payload = {
        "notebook": "NN_06_film_ablation.ipynb", "git_commit": rd.git_commit(), "config_identity": cfg.identity(),
        "slice_fingerprint": bundle.fingerprint, "device": str(DEVICE), "seeds": list(nc.SEEDS),
        "protocol": NN05["protocol"], "verdict": VERDICT,
        "paired_test": {"mean_diff": float(D.mean()), "std_diff": float(D.std(ddof=1)), "t": float(t_stat), "p": float(p_value), "n_cycles": int(len(D))},
        "results": RESULTS,
    }
    out_path.write_text(json.dumps(payload))
    print("wrote", out_path, f"({out_path.stat().st_size / 1e6:.2f} MB)")

wrote /Users/gavinhussey/Downloads/AtlasQuant/neural_ranker/outputs/NN_06_film_ablation.json (2.40 MB)


## Stop condition

`BUILD_SPEC.md` §9: *"If `NN_06` shows FiLM does nothing, say so plainly and stop. That is the answer to the question this experiment was built to ask."*

This notebook's result is **null on this slice** (verdict above, by the rule registered before the run). Accordingly, `NN_07` (seed variance) and `NN_08` (baseline comparison and the `0.5·LGBM + 0.5·NN` ensemble) are **not pursued now**: both are performance-seeking follow-ups whose premise — a conditioning effect worth characterizing and comparing — was not established, and pursuing them would turn a null hypothesis test into a model-improvement exercise. The seed-variance numbers this ablation needed are already reported above. `NN_09` consolidates the record from the saved `NN_05`/`NN_06` artifacts without any new training, metric or claim. No change to the architecture, early stopping, target temperature, seeds or feature policy is made after this point.

## What this established

**Experiment answer, stated plainly (`BUILD_SPEC.md` §9): null on this slice.** FiLM macro conditioning does not add out-of-sample IC under the fixed protocol on the cached 2020-10-01 .. 2024-01-01 slice: the paired FiLM − no-FiLM difference is −0.0094 IC (p = 0.244, FiLM higher in 4 of 8 cycles). **No production value is claimed for either arm.** Per §9 the series stops here (see *Stop condition* above).

Every number below was printed by a cell above in this executed run; all fits ran on CPU and are bit-reproducible.

**The ablation is exactly paired.** `NN_05`'s artifact was verified against the slice before use (same fingerprint and config identity; baseline IC recomputed from its returns and baseline scores equal to the stored IC bit for bit on all 8 measured cycles; 0 Materials names; scored counts 1,370 .. 1,401). With the same seed the two arms start with all 16 parameter tensors bit-identical; at initialization max|γ| = 3.53e-03, max|β| = 3.57e-03, and the arms' first scores differ by at most 2.74e-04. 90 fits (9 cycles × 5 seeds × 2 arms) in 451 s, mean 5.0 s per fit. **The FiLM arm reproduced `NN_05`'s saved records with 0 mismatches** — same IC, best validation loss, best step, steps and scores on all 45 fits. 88 IC values, every one through `spearman_correlation`; the NN's measured cycles are the baseline's 8 exactly.

**Per-cycle result (seed-means over the five seeds; the difference is FiLM − no-FiLM on the same cycle, names, returns and seed).**

| cycle | LightGBM | FiLM | no-FiLM | FiLM − no-FiLM | diff min (seeds) | diff max (seeds) | seeds with FiLM higher |
|---|---|---|---|---|---|---|---|
| 2022-01-01 | −0.0261 | +0.0759 | +0.1179 | −0.0420 | −0.1231 | −0.0062 | 0 of 5 |
| 2022-04-01 | −0.1900 | −0.1422 | −0.1002 | −0.0420 | −0.1115 | −0.0027 | 0 of 5 |
| 2022-07-01 | +0.0618 | +0.1046 | +0.1044 | +0.0002 | −0.0085 | +0.0070 | 3 of 5 |
| 2022-10-01 | +0.0583 | +0.0633 | +0.0641 | −0.0008 | −0.0089 | +0.0036 | 3 of 5 |
| 2023-01-01 | +0.0708 | +0.0584 | +0.0473 | +0.0111 | −0.0454 | +0.0999 | 2 of 5 |
| 2023-04-01 | +0.0335 | +0.0874 | +0.0974 | −0.0099 | −0.0700 | +0.0232 | 2 of 5 |
| 2023-07-01 | +0.0877 | +0.1029 | +0.0985 | +0.0044 | −0.0022 | +0.0128 | 4 of 5 |
| 2023-10-01 | +0.1240 | +0.0436 | +0.0399 | +0.0037 | −0.0231 | +0.0203 | 3 of 5 |

**Primary paired test across the 8 cycles** (seed-mean FiLM IC vs seed-mean no-FiLM IC, df = 7, two-sided): **mean difference −0.0094**, std 0.0210, SE 0.0074, **t = −1.271, p = 0.244**, FiLM higher in 4 of 8 cycles.

**Per seed** (8 cycles each):

| seed | mean FiLM IC | mean no-FiLM IC | mean diff | t | p | cycles with FiLM higher |
|---|---|---|---|---|---|---|
| 0 | +0.0498 | +0.0561 | −0.0062 | −0.956 | 0.371 | 2 of 8 |
| 1 | +0.0331 | +0.0605 | −0.0274 | −1.643 | 0.144 | 4 of 8 |
| 2 | +0.0522 | +0.0760 | −0.0238 | −1.702 | 0.133 | 3 of 8 |
| 3 | +0.0637 | +0.0682 | −0.0045 | −0.457 | 0.661 | 3 of 8 |
| 4 | +0.0474 | +0.0325 | +0.0149 | +1.172 | 0.280 | 5 of 8 |

**Seed variance.** The per-seed mean difference runs from −0.0274 to +0.0149 (std across seeds 0.0170); within a cycle, the difference's std across seeds averages 0.0276. The mean effect is 0.55 times the seed-noise scale. Three of five seeds lean against FiLM, one for it, none significantly.

**Both arms behave identically under early stopping.** FiLM: best step 10–50 (median 20), stopped at 210–250 steps, 0 at the cap; no-FiLM: best step 10–50 (median 20), stopped at 210–250, 0 at the cap. The FiLM path was not inert: at the scored target, max|γ| across the 45 FiLM fits ran from 0.4106 to 1.7745 (median 0.9046), so the conditioner did modulate the representation substantially even in 10–50 steps — it just did not improve out-of-sample IC.

**Verdict, by the rule registered before the run: NULL on this slice: the FiLM effect is not distinguishable from zero at 8 cycles.**

**What this does not establish.** Eight paired cycles is thin (the full-history baseline has 22), the conditioner saw only 4–12 distinct macro states per training window (`NN_03`), and both arms are 10–50-step models under the fixed protocol (`NN_05`). A null at this sample size means "not distinguishable from zero here", not "macro conditioning is useless"; with seed noise of 0.0170 on the per-seed mean and an SE of 0.0074 across cycles, an effect smaller than roughly ±0.02 IC could not have been detected.

**Point-in-time, checked on all 90 records.** Every training and validation quarter is strictly before its target and inside the trailing window, every label used was available strictly before the target's cutoff, the validation quarter is the most recent quarter of the production window, and the target quarter never appears in the window.

**Saved.** `neural_ranker/outputs/NN_06_film_ablation.json` (2.40 MB, gitignored): both arms' per-fit records with scores for the measured cycles, the paired-test summary and the verdict. `NN_09` consolidates this and `NN_05`'s artifact for transcription into `reproducibility_findings.md`.